# 🤖 Treino, Validação e Otimização

- Divisão **70/15/15 estratificada e agrupada por `md5`** (nenhum ficheiro aparece em dois conjuntos).
- 7 classificadores comparados no Validation Set e com **Stratified Group 5-Fold CV** no treino.
- Métrica principal: **Recall** da classe Malware, mas só entre modelos com CV F1 ≥ 0,95 (senão o Naive Bayes, que classifica tudo como malware, ganhava).
- `GridSearchCV` do XGBoost orientado ao Recall; em caso de empate mantém-se o baseline.

In [1]:
import sys, os, warnings
sys.path.append('../src')
warnings.filterwarnings('ignore')
import pandas as pd
from features import prepare_datasets
from models import (split_dataset, evaluate_all_models, tune_xgboost, choose_baseline_or_tuned,
                    overfitting_report, save_model)

df = pd.read_csv('../data/processed/ransom_cleaned.csv')
X, y, le, groups = prepare_datasets(df, return_groups=True)
print(X.shape, list(le.classes_))
X_train, X_val, X_test, y_train, y_val, y_test, g_train = split_dataset(X, y, groups)

(21752, 165) ['Benign', 'Malware']
🔹 Treino: 15220 amostras (70.0%) | % Malware = 50.0%
🔹 Validação: 3267 amostras (15.0%) | % Malware = 50.0%
🔹 Teste: 3265 amostras (15.0%) | % Malware = 50.0%
🔹 Ficheiros (md5) partilhados entre conjuntos: 0


In [2]:
tabela, ajustados = evaluate_all_models(X_train, y_train, X_val, y_val, g_train)
tabela.round(4)

✔️ LogisticRegression  CV Recall 0.9833 | CV F1 0.9858 | Val Recall 0.9835 | Val F1 0.9865
✔️ DecisionTree        CV Recall 0.9895 | CV F1 0.9899 | Val Recall 0.9914 | Val F1 0.9929
✔️ KNN                 CV Recall 0.9055 | CV F1 0.9083 | Val Recall 0.9081 | Val F1 0.9171
✔️ NaiveBayes          CV Recall 0.9983 | CV F1 0.6927 | Val Recall 0.9976 | Val F1 0.6895
✔️ LinearSVM           CV Recall 0.9908 | CV F1 0.9898 | Val Recall 0.9914 | Val F1 0.9905
✔️ RandomForest        CV Recall 0.9966 | CV F1 0.9945 | Val Recall 0.9945 | Val F1 0.9930
✔️ XGBoost             CV Recall 0.9971 | CV F1 0.9949 | Val Recall 0.9969 | Val F1 0.9951


,Model,CV_Recall,CV_F1,Val_Recall,Val_Precision,Val_F1,Train_Acc,Val_Acc,Elegivel
0,XGBoost,0.9971,0.9949,0.9969,0.9933,0.9951,1.0000,0.9951,True
1,RandomForest,0.9966,0.9945,0.9945,0.9915,0.9930,1.0000,0.9930,True
2,LinearSVM,0.9908,0.9898,0.9914,0.9896,0.9905,0.9928,0.9905,True
3,DecisionTree,0.9895,0.9899,0.9914,0.9945,0.9929,1.0000,0.9930,True
4,LogisticRegression,0.9833,0.9858,0.9835,0.9895,0.9865,0.9908,0.9865,True
5,NaiveBayes,0.9983,0.6927,0.9976,0.5268,0.6895,0.5570,0.5510,False
6,KNN,0.9055,0.9083,0.9081,0.9263,0.9171,0.9497,0.9180,False


In [3]:
melhor = tabela.iloc[0]['Model']
print('Melhor modelo:', melhor)
otimizado, params, resultados_grid = tune_xgboost(X_train, y_train, g_train)
modelo_final, versao = choose_baseline_or_tuned(ajustados['XGBoost'], otimizado, X_val, y_val)
print('Modelo final:', versao)

Melhor modelo: XGBoost
🔧 Melhores parâmetros: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100, 'reg_lambda': 5} | CV Recall 0.9979 | CV F1 0.9922
Validação | baseline (recall, f1) = 0.9969, 0.9951 | otimizado = 0.9963, 0.9921
Modelo final: baseline


In [4]:
ajustados['XGBoost (otimizado)'] = otimizado
overfitting_report(ajustados, X_train, y_train, X_val, y_val).round(4)

,Model,Acc_Treino,Acc_Validação,Diferença_pp
0,LogisticRegression,0.9908,0.9865,0.4270
1,DecisionTree,1.0000,0.9930,0.7040
2,KNN,0.9497,0.9180,3.1770
3,NaiveBayes,0.5570,0.5510,0.6066
4,LinearSVM,0.9928,0.9905,0.2261
5,RandomForest,1.0000,0.9930,0.7040
6,XGBoost,1.0000,0.9951,0.4897
7,XGBoost (otimizado),0.9928,0.9920,0.0797


O modelo só é guardado aqui; o **Test Set não foi usado** até agora (é avaliado uma única vez no notebook 04).

In [5]:
save_model(modelo_final, X.columns, '../models', meta={'modelo': melhor, 'versao': versao})
print('✅ Modelo guardado em ../models/ransomware_model.pkl')

✅ Modelo guardado em ../models/ransomware_model.pkl
